In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime
import os
import urllib3
import re  # 정규표현식 라이브러리 추가

# SSL 인증서 관련 오류 방지
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['CURL_CA_BUNDLE'] = ''

# 1. 서울 구 리스트
seoul_gu_list = [
    "강남구", "강동구", "강북구", "강서구", "관악구", "광진구", "구로구", "금천구",
    "노원구", "도봉구", "동대문구", "동작구", "마포구", "서대문구", "서초구", "성동구",
    "성북구", "송파구", "양천구", "영등포구", "용산구", "은평구", "종로구", "중구", "중랑구"
]

def collect_mega_coffee():
    all_stores = []
    base_url = "https://www.mega-mgccoffee.com/store/find/store_search.php"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print(f"데이터 수집 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("-" * 50)

    for gu in seoul_gu_list:
        params = {"store_search": gu}
        try:
            response = requests.get(base_url, params=params, headers=headers, verify=False)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            li_list = soup.select('li > a > div.cont_text')
            
            for item in li_list:
                # 5. 매장명 파싱
                name_div = item.find('div', class_='cont_text_inner')
                store_name = name_div.find('b').get_text(strip=True) if name_div and name_div.find('b') else "N/A"
                
                # 6~7. 주소 및 전화번호 분리 로직 (02- 및 070- 대응)
                info_div = item.find('div', class_='cont_text_info')
                if info_div:
                    raw_text = info_div.get_text().strip()
                    
                    # 정규표현식으로 02- 또는 070- 이 시작되는 위치를 찾음
                    # r'(02-|070-)'는 "02-" 또는 "070-" 중 먼저 걸리는 것을 의미함
                    match = re.search(r'(02-|070-)', raw_text)
                    
                    if match:
                        split_idx = match.start() # 찾은 패턴이 시작되는 인덱스 번호
                        address = raw_text[:split_idx].strip()
                        phone = raw_text[split_idx:].strip()
                    else:
                        # 전화번호 패턴이 발견되지 않은 경우
                        address = raw_text
                        phone = ""

                    # 10. 주소가 "서울"로 시작하는 데이터만 포함
                    if address.startswith("서울"):
                        all_stores.append({
                            "매장명": store_name,
                            "주소": address,
                            "전화번호": phone
                        })
            
            # 8. 한 구가 끝날 때마다 시간 기록
            current_time = datetime.now().strftime('%H:%M:%S')
            print(f"[{current_time}] {gu} 완료 (현재 누적 매장 수: {len(all_stores)}개)")
            
            time.sleep(0.6)
            
        except Exception as e:
            print(f"{gu} 수집 중 에러 발생: {e}")

    # 10. 중복 데이터 제거
    df = pd.DataFrame(all_stores)
    df = df.drop_duplicates(subset=['매장명', '주소'], keep='first')

    # 9. CSV 저장
    df.to_csv("mega_coffee.csv", index=False, encoding="utf-8-sig")
    print("-" * 50)
    print(f"최종 수집 완료! 총 {len(df)}개 매장이 'mega_coffee.csv' 파일로 저장되었습니다.")

if __name__ == "__main__":
    collect_mega_coffee()

데이터 수집 시작: 2026-03-28 14:52:41
--------------------------------------------------
[14:52:41] 강남구 완료 (현재 누적 매장 수: 85개)
[14:52:42] 강동구 완료 (현재 누적 매장 수: 129개)
[14:52:43] 강북구 완료 (현재 누적 매장 수: 155개)
[14:52:44] 강서구 완료 (현재 누적 매장 수: 209개)
[14:52:44] 관악구 완료 (현재 누적 매장 수: 249개)
[14:52:45] 광진구 완료 (현재 누적 매장 수: 285개)
[14:52:46] 구로구 완료 (현재 누적 매장 수: 327개)
[14:52:47] 금천구 완료 (현재 누적 매장 수: 357개)
[14:52:47] 노원구 완료 (현재 누적 매장 수: 401개)
[14:52:48] 도봉구 완료 (현재 누적 매장 수: 428개)
[14:52:49] 동대문구 완료 (현재 누적 매장 수: 466개)
[14:52:50] 동작구 완료 (현재 누적 매장 수: 488개)
[14:52:50] 마포구 완료 (현재 누적 매장 수: 540개)
[14:52:51] 서대문구 완료 (현재 누적 매장 수: 568개)
[14:52:52] 서초구 완료 (현재 누적 매장 수: 616개)
[14:52:53] 성동구 완료 (현재 누적 매장 수: 648개)
[14:52:53] 성북구 완료 (현재 누적 매장 수: 689개)
[14:52:54] 송파구 완료 (현재 누적 매장 수: 749개)
[14:52:55] 양천구 완료 (현재 누적 매장 수: 790개)
[14:52:56] 영등포구 완료 (현재 누적 매장 수: 848개)
[14:52:57] 용산구 완료 (현재 누적 매장 수: 867개)
[14:52:58] 은평구 완료 (현재 누적 매장 수: 902개)
[14:52:59] 종로구 완료 (현재 누적 매장 수: 935개)
[14:53:00] 중구 완료 (현재 누적 매장 수: 972개)
[14:53:00] 중랑구 완료 (현재 누적 매장 수

In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime
import os
import urllib3
import re

# SSL 인증서 및 경고 무시 설정
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['CURL_CA_BUNDLE'] = ''

# 1. 제주특별자치도 시 리스트 (2개 지역)
jeju_list = ["제주시", "서귀포시"]

def collect_mega_jeju():
    new_stores = []
    base_url = "https://www.mega-mgccoffee.com/store/find/store_search.php"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # 마스터 전화번호 패턴 (제주 064 포함 전지역 대응)
    master_phone_regex = r'(02-|03[1-3]-|04[1-4]-|05[1-5]-|06[1-4]-|070-|050\d?-|15\d{2}-|16\d{2}-|18\d{2}-|010-)'

    print(f"제주특별자치도 데이터 수집 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("-" * 50)

    for area in jeju_list:
        # 지역명만 사용하여 검색 (제주 제주시 X -> 제주시 O)
        params = {"store_search": area}
        try:
            response = requests.get(base_url, params=params, headers=headers, verify=False)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            li_list = soup.select('li > a > div.cont_text')
            
            for item in li_list:
                # 5. 매장명 파싱
                name_div = item.find('div', class_='cont_text_inner')
                store_name = name_div.find('b').get_text(strip=True) if name_div and name_div.find('b') else "N/A"
                
                # 6~7. 주소 및 전화번호 분리
                info_div = item.find('div', class_='cont_text_info')
                if info_div:
                    raw_text = info_div.get_text().strip()
                    
                    # 마스터 패턴으로 전화번호 시작 위치 찾기
                    match = re.search(master_phone_regex, raw_text)
                    
                    if match:
                        split_idx = match.start()
                        address = raw_text[:split_idx].strip()
                        phone = raw_text[split_idx:].strip()
                    else:
                        address = raw_text
                        phone = ""

                    # 10. 주소가 "제주"로 시작하는 데이터만 필터링
                    if address.startswith("제주"):
                        new_stores.append({
                            "매장명": store_name,
                            "주소": address,
                            "전화번호": phone
                        })
            
            # 8. 한 지역이 끝날 때마다 시간 기록
            current_time = datetime.now().strftime('%H:%M:%S')
            print(f"[{current_time}] {area} 완료 (누적 제주 매장: {len(new_stores)}개)")
            
            time.sleep(0.5)
            
        except Exception as e:
            print(f"{area} 수집 중 에러 발생: {e}")

    # --- 최종 파일 병합 및 중복 제거 ---
    file_name = "mega_coffee.csv"
    df_new = pd.DataFrame(new_stores)
    
    if os.path.exists(file_name):
        df_old = pd.read_csv(file_name)
        # 지금까지 모은 모든 지역 데이터와 병합
        df_final = pd.concat([df_old, df_new], ignore_index=True)
        print(f"\n기존 전국 데이터에 제주도 매장 {len(df_new)}개를 추가합니다.")
    else:
        df_final = df_new
        print(f"\n기존 파일이 없어 새로운 파일을 생성합니다.")

    # 10. 전체 데이터 중복 제거 (최종 점검)
    df_final = df_final.drop_duplicates(subset=['매장명', '주소'], keep='first')

    # 9. CSV로 최종 저장
    df_final.to_csv(file_name, index=False, encoding="utf-8-sig")
    print("-" * 50)
    print("🎉 축하합니다! 대한민국 전역의 메가커피 데이터 수집이 완료되었습니다.")
    print(f"최종 저장된 전국의 총 매장 수: {len(df_final)}개")
    print(f"파일 경로: {os.path.abspath(file_name)}")

if __name__ == "__main__":
    collect_mega_jeju()

제주특별자치도 데이터 수집 시작: 2026-03-28 15:14:17
--------------------------------------------------
[15:14:17] 제주시 완료 (누적 제주 매장: 33개)
[15:14:18] 서귀포시 완료 (누적 제주 매장: 48개)

기존 전국 데이터에 제주도 매장 48개를 추가합니다.
--------------------------------------------------
🎉 축하합니다! 대한민국 전역의 메가커피 데이터 수집이 완료되었습니다.
최종 저장된 전국의 총 매장 수: 4039개
파일 경로: c:\PYTHON\crawling_archive\mega_coffee.csv
